# Day 12 — NLP & Classical Text Processing

## RetailMind AI Increment

### Objective

Build a classical sentiment-analysis pipeline for Amazon product reviews.

Pipeline:

`Review → Cleaning → Tokenization → Stopword Removal → TF-IDF → Logistic Regression → Sentiment`

## 1. Import Libraries

In [ ]:
import re
import pickle
import pandas as pd
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

## 2. Download Required NLTK Resources

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

## 3. Load Amazon Reviews

Save your Amazon reviews dataset as:

`amazon_reviews.csv`

The dataset should contain review text and a sentiment label.

In [ ]:
df = pd.read_csv('amazon_reviews.csv')

print(df.shape)
df.head()

## 4. Inspect Dataset

In [ ]:
print(df.columns)
print(df.info())
print(df.isnull().sum())

## 5. Select Review and Label Columns

Change these names if your dataset uses different column names.

In [ ]:
TEXT_COLUMN = 'content'
LABEL_COLUMN = 'label'

df = df[[TEXT_COLUMN, LABEL_COLUMN]].copy()

df.columns = ['review', 'sentiment']

df.head()

## 6. Handle Missing Values

In [ ]:
df['review'] = df['review'].fillna('')

print(df.isnull().sum())

## 7. Prepare Stopwords

In [ ]:
stop_words = set(stopwords.words('english'))

print(list(stop_words)[:20])

## 8. Text Cleaning Function

The function performs:

- Lowercasing
- URL removal
- HTML removal
- Number and punctuation removal
- Tokenization
- Stopword removal

In [ ]:
def clean_text(text):
    text = str(text).lower()

    text = re.sub(r'https?://\\S+|www\\.\\S+', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-zA-Z\\s]', ' ', text)

    tokens = word_tokenize(text)

    tokens = [
        word for word in tokens
        if word not in stop_words and len(word) > 1
    ]

    return ' '.join(tokens)

## 9. Test Cleaning Function

In [ ]:
sample_review = 'This product is AMAZING!!! I really loved it. https://example.com'

print('Original:')
print(sample_review)

print('\nCleaned:')
print(clean_text(sample_review))

## 10. Clean All Reviews

In [ ]:
df['cleaned_review'] = df['review'].apply(clean_text)

df[['review', 'cleaned_review']].head()

## 11. Define X and y

In [ ]:
X = df['cleaned_review']
y = df['sentiment']

print('X shape:', X.shape)
print('y shape:', y.shape)

## 12. Train/Test Split

Split before fitting TF-IDF so information from the test set does not leak into training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## 13. Build TF-IDF + Logistic Regression Pipeline

`ngram_range=(1, 2)` allows both individual words and two-word phrases.

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        min_df=2
    )),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

pipeline

## 14. Train Model

In [ ]:
pipeline.fit(X_train, y_train)

print('Training completed.')

## 15. Make Predictions

In [ ]:
y_pred = pipeline.predict(X_test)

print(y_pred[:10])

## 16. Evaluate Model

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print('Accuracy :', round(accuracy, 4))
print('Precision:', round(precision, 4))
print('Recall   :', round(recall, 4))
print('F1 Score :', round(f1, 4))

## 17. Classification Report

In [ ]:
print(classification_report(y_test, y_pred, zero_division=0))

## 18. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

## 19. Most Informative Terms

Because Logistic Regression assigns a coefficient to each TF-IDF feature, we can inspect which terms push predictions toward each sentiment.

In [ ]:
tfidf = pipeline.named_steps['tfidf']
classifier = pipeline.named_steps['classifier']

feature_names = tfidf.get_feature_names_out()
coefficients = classifier.coef_[0]

term_scores = pd.DataFrame({
    'term': feature_names,
    'score': coefficients
})

term_scores.head()

## 20. Most Positive Terms

In [ ]:
positive_terms = term_scores.sort_values(
    by='score',
    ascending=False
).head(20)

positive_terms

## 21. Most Negative Terms

In [ ]:
negative_terms = term_scores.sort_values(
    by='score',
    ascending=True
).head(20)

negative_terms

## 22. Predict New Customer Reviews

In [ ]:
new_reviews = [
    'This product is excellent and works perfectly',
    'Very poor quality and completely useless'
]

cleaned_new_reviews = [clean_text(review) for review in new_reviews]

predictions = pipeline.predict(cleaned_new_reviews)

for review, prediction in zip(new_reviews, predictions):
    print('Review:', review)
    print('Prediction:', prediction)
    print()

## 23. Save the Review Processing Model

The complete pipeline contains both TF-IDF and Logistic Regression, so the same feature transformation will be applied after loading the model.

In [ ]:
with open('review_sentiment_model.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

print('Model saved successfully.')

## 24. Load Saved Model

In [ ]:
with open('review_sentiment_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

print('Model loaded successfully.')

## 25. Test Loaded Model

In [ ]:
test_review = 'Amazing product and excellent value for money'

cleaned_test_review = clean_text(test_review)

prediction = loaded_model.predict([cleaned_test_review])

print('Review:', test_review)
print('Prediction:', prediction[0])

## 26. Day 12 Summary

In this RetailMind increment we built:

- Amazon review loading
- Text cleaning
- Lowercasing
- Noise removal
- Tokenization
- Stopword removal
- TF-IDF vectorization
- Logistic Regression sentiment classification
- Accuracy, precision, recall and F1 evaluation
- Informative positive and negative term extraction
- New-review prediction
- Model persistence

A later increment can replace this classical NLP baseline with BERT.